In [ ]:
# ML Modelleri Notebook'u
from sklearn.model_selection import train_test_split

In [ ]:
# 1. GEREKSİZ SÜTUNLARI SİLME
silinecek_sutunlar = ['zaman_damgasi', 'anket_hedefi']
df_oyuncular = df_oyuncular.drop(columns=silinecek_sutunlar, errors='ignore')

# 2. DÖNÜŞÜM SÖZLÜKLERİ (Mapping Dictionaries)
# Not: Modeller için düşük rakam = düşük şiddet/sıklık, yüksek rakam = yüksek şiddet/sıklık mantığı kurulmuştur.

donusum_haritasi = {
    'oyuncu_cinsiyet': {
        'Kadın': 0, 'Erkek': 1, 'Belirtmek İstemiyorum': 2
    },
    'oyuncu_yas': {
        '18-22': 0, '23-28': 1, '29-35': 2, '36-45': 3, '46-55': 4, '55+': 5
    },
    'oyuncu_haftalik_oyun_gunu': {
        'Hiçbir zaman': 0, 'Nadiren (Ayda 1-2 kez)': 1, 'Bazen (Haftada 1-2 kez)': 2, 
        'Sıklıkla (Haftada 3-4 kez)': 3, 'Neredeyse her gün / Her gün': 4
    },
    'oyuncu_gunluk_sure': {
        '1 saatten az': 0, '1-3 saat': 1, '3-5 saat': 2, '5 saatten fazla': 3
    },
    'oyuncu_harcama_butcesi': {
        'Hiç harcama yapmam': 0, "250 TL'den az": 1, '250 - 1000 TL arası': 2, '1000 TL ve üzeri': 3
    },
    'oyuncu_harcama_etkisi': {
        'Hiç harcamam veya sınırlarımı asla aşmam.': 0, 
        'Bazen limitimi aşarım ama bütçemi sarsmaz.': 1, 
        'Bütçemi zorlar, diğer ihtiyaçlarımdan kısmam gerekir.': 2, 
        'Sonradan pişman olurum veya harcamalarımı çevremden gizlerim.': 3
    },
    'oyuncu_maglubiyet_tepkisi': {
        'Sakince oyunu kapatır veya başka bir işe geçerim.': 0, 
        'Biraz moralim bozulur ama oyuna ara veririm.': 1, 
        'Sadece kaybımı telafi etmek için bir maç daha atar, sonra çıkarım.': 2, 
        'Hırslanırım ve mutlaka kazanana kadar peş peşe yeni maçlara girmeye devam ederim.': 3,
        'Sinirimden saatlerce bilgisayarın/telefonun başından kalkamam.': 4
    },
    'oyuncu_stres_etkisi': {
        'Tamamen rahatlatır, stresimi alır.': 0, 
        'Oynadığım sürece rahatlarım, kafam dağıtır.': 1, 
        'Nötr (Oynamak veya oynamamak genel ruh halimi pek değiştirmez).': 2, 
        'Zorlu maçlarda veya rekabet gerektiren anlarda zaman zaman stres yaşadığım olur.': 3,
        'Oynayamadığım zamanlarda çabuk canım sıkılır veya huzursuz/gergin hissederim, .': 4 # Formdaki sonundaki virgül hatası dahil edildi
    },
    'oyuncu_uyku_feragati': {
        'Hiçbir zaman': 0, 'Nadiren (Ayda 1-2 kez)': 1, 'Bazen (Ayda birkaç kez)': 2, 
        'Sıklıkla (Haftada birkaç kez)': 3, 'Neredeyse her gün': 4
    },
    'oyuncu_ruh_hali': {
        'Hiç etkilemez, anında unuturum.': 0, 'Sadece anlık olarak canım sıkılır.': 1, 
        'O günkü genel moralimi düşürür.': 2, 'Çevreme karşı gergin ve tahammülsüz olurum.': 3, 
        'Günlük hayatta öfke patlamaları yaşamama sebep olur.': 4
    },
    'oyuncu_pegi_dikkat': {
        'Evet, her zaman': 0, 'Bazen': 1, 'Hayır, dikkat etmiyorum': 2, 'PEGI nedir bilmiyorum': 3
    },
    'oyuncu_platform_kisitlama_gorus': {
        'Kesinlikle desteklemiyorum': 0, 'Desteklemiyorum': 1, 'Kararsızım': 2, 
        'Destekliyorum': 3, 'Kesinlikle destekliyorum': 4
    },
    'veli_ebeveyn_onay_gorus': { # df_oyuncular içinde kalan ortak soru
        'Kesinlikle desteklemiyorum': 0, 'Desteklemiyorum': 1, 'Kararsızım': 2, 
        'Destekliyorum': 3, 'Kesinlikle destekliyorum': 4
    }
}

# 3. DÖNÜŞÜMÜ UYGULAMA
df_oyuncular = df_oyuncular.replace(donusum_haritasi)

print("✅ Sayısal dönüşüm başarıyla tamamlandı!")
display(df_oyuncular.head(2))

In [ ]:
# Literatürden hesapladığımız Risk Katsayıları 
# (UYARI: Bu metinler tablodaki sütun adlarıyla harfi harfine aynı olmalı!)
risk_weights = {
    "Çocuğunuz ortalama bir haftada, haftanın kaç günü oyun oynar?": 0.82,
    "Çocuğunuz oyun oynadığı günlerde (ekran başına geçtiğinde) genellikle toplam ne kadar süre oyun oynar?": 0.76,
    "Çocuğunuzun en çok vakit geçirdiği oyun türü genelde hangisidir?": 0.64,
    "Çocuğunuzun oyun içi satın alımlar için ayda ortalama ne kadarlık bir harcama yapmasına izin veriyorsunuz (veya kendi harçlığından harcıyor)?": 0.68,
    "Çocuğunuzun oyun içi harcamaları bütçenizi ve aranızdaki güveni nasıl etkiliyor?": 0.68
}

def calculate_risk_score(row):
    total_risk = 0.0
    
    # Sözlükteki her bir sütun ve ağırlık için döngü başlat
    for col_name, weight in risk_weights.items():
        if col_name in row.index:
            # Cevabı kontrol et (Hata almamak için string'e çeviriyoruz)
            answer = str(row[col_name]).lower()
            
            # Eğer cevabın içinde "bilmiyorum" kelimesi geçiyorsa cezayı kes
            if "bilmiyorum" in answer:
                total_risk += weight
                
    return total_risk

# apply metodu ile fonksiyonu tüm canlı veriye uygula
df_oyuncular['gozetimsizlik_risk_skoru'] = df_oyuncular.apply(calculate_risk_score, axis=1)

# Skoru hesaplanmış ilk 5 veriyi görelim
display(df_oyuncular[['gozetimsizlik_risk_skoru']])

In [ ]:
# 4. FAKTÖR SKORLARI (Feature Engineering)

faktor_yukleri = {
    'Faktor_1_Bağımlılık': {
        'oyuncu_gunluk_sure': 0.75,
        'oyuncu_uyku_feragati': 0.68,
        'oyuncu_stres_etkisi': 0.62
    },
    'Faktor_2_Bilinç': {
        'oyuncu_pegi_dikkat': 0.81,
        'veli_ebeveyn_onay_gorus': 0.74
    }
}

# Faktör skorlarını hesaplama
for faktor_adi, sutunlar in faktor_yukleri.items():
    # Dinamik sütun ismi oluştur (örn: 'faktor_1_bağımlılık_skoru')
    skor_sutun_adi = faktor_adi.lower() + '_skoru'
    
    # İlk olarak sıfır değeriyle başla
    skor = pd.Series(0.0, index=df_oyuncular.index)
    
    # Sözlükteki sütunları döngüyle işle
    for sutun, yuk in sutunlar.items():
        if sutun in df_oyuncular.columns:
            # Sütun değerini ağırlıkla çarpıp skora ekle
            skor += df_oyuncular[sutun] * yuk
    
    # Veri setine ekle
    df_oyuncular[skor_sutun_adi] = skor

# Hesaplanan faktör skorlarını göster
print("✅ Faktör skorları başarıyla hesaplandı!")
display(df_oyuncular[[col for col in df_oyuncular.columns if '_skoru' in col]].head())